# Dynamic Tool Filtering

This notebook isolates the retrieval-based filtering step. It uses a lightweight embedding model to drop irrelevant tool options, creating a cleaner, smaller dataset for the LLM to process.

In [ ]:
# === IMPORT LIBRARIES ===
import os
import json
import torch
from sentence_transformers import SentenceTransformer, util

In [ ]:
# === CONFIGURATION ===
MODEL_ID = "BAAI/bge-small-en-v1.5"
TOP_K = 10

In [ ]:
# Ensure output directories exist
os.makedirs("../output/cache", exist_ok=True)

# Load lightweight embedding model
print("Loading embedding model...")
retriever = SentenceTransformer(MODEL_ID)

In [ ]:
def filter_options_for_sample(sample, top_k=TOP_K):
    user_context = sample["full_context"]
    options_dict = sample["options"]
    
    if len(options_dict) <= top_k:
        return sample # No need to filter if options are already small
        
    option_letters = list(options_dict.keys())
    tool_descriptions = [
        f"{tool['name']}: {tool.get('description', '')}" 
        for tool in options_dict.values()
    ]
    
    query_emb = retriever.encode(user_context, convert_to_tensor=True)
    tools_emb = retriever.encode(tool_descriptions, convert_to_tensor=True)
    cosine_scores = util.cos_sim(query_emb, tools_emb)[0]
    
    top_results = torch.topk(cosine_scores, k=top_k)
    top_indices = top_results.indices.tolist()
    
    filtered_options = {}
    new_letters = [chr(ord('A') + i) for i in range(top_k)]
    
    correct_original_letter = sample.get("answer")
    new_answer = None
    
    for new_idx, original_idx in enumerate(top_indices):
        original_letter = option_letters[original_idx]
        new_letter = new_letters[new_idx]
        filtered_options[new_letter] = options_dict[original_letter]
        
        # Track the ground truth answer if it exists
        if original_letter == correct_original_letter:
            new_answer = new_letter
            
    # Note: If the true answer gets filtered out (a retrieval error), 
    # it will be 'None'. In a real pipeline, you'd force-inject the true answer for training data.
    sample["options"] = filtered_options
    if new_answer:
        sample["answer"] = new_answer
        
    return sample

In [ ]:
def process_and_cache(input_path, output_path, top_k=TOP_K):
    print(f"Filtering {input_path}...")
    filtered_dataset = []
    
    with open(input_path, "r") as f:
        for line in f:
            sample = json.loads(line)
            filtered_sample = filter_options_for_sample(sample, top_k=top_k)
            filtered_dataset.append(filtered_sample)
            
    with open(output_path, "w") as f:
        for sample in filtered_dataset:
            f.write(json.dumps(sample) + "\n")
            
    print(f"Saved {len(filtered_dataset)} samples to {output_path}!")

In [ ]:
# === MAIN ===
safe_model_id = MODEL_ID.replace("/", "-")

process_and_cache("../data/test.jsonl", 
                  f"../output/cache/test_filtered_top{TOP_K}_{safe_model_id}.jsonl", 
                  top_k=TOP_K)

process_and_cache("../data/train.jsonl", 
                  f"../output/cache/train_filtered_top{TOP_K}_{safe_model_id}.jsonl", 
                  top_k=TOP_K)

## SFT Dataset Preparation & Splitting
Now that we have filtered the options down to the top K, we need to prepare the exact files required for SFT training (`02_sft_training.ipynb`). This involves:
1. **Cleaning:** Dropping samples where the ground-truth answer was lost during retrieval.
2. **Formatting:** Stripping tool descriptions to create the `structural` configuration.
3. **Splitting:** Performing an 85/15 stratified train/validation split.

In [ ]:
import copy
from sklearn.model_selection import train_test_split

FILTERED_TRAIN_PATH = f"../output/cache/train_filtered_top{TOP_K}_{safe_model_id}.jsonl"
DEV_PATH = "../data/addition.jsonl"
DATA_DIR = "../output/cache"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def strip_descriptions(obj):
    if isinstance(obj, dict):
        new_dict = {}
        for k, v in obj.items():
            if k == "description":
                continue
            new_dict[k] = strip_descriptions(v)
        return new_dict
    elif isinstance(obj, list):
        return [strip_descriptions(item) for item in obj]
    else:
        return obj

def make_structural(data):
    structural = []
    for row in data:
        new_row = copy.deepcopy(row)
        new_row["options"] = strip_descriptions(new_row["options"])
        structural.append(new_row)
    return structural

In [ ]:
print("Loading dev data to map domains...")
dev_data = load_jsonl(DEV_PATH)
tool_to_domain = {}
for item in dev_data:
    domain = item.get("domain", "Unknown")
    for tool in item["tools"]:
        tool_to_domain[tool["name"]] = domain

def get_domain(row):
    correct_tool = row["options"][row["answer"]]
    return tool_to_domain.get(correct_tool["name"], "Unknown")

print("Loading filtered training data...")
train_filtered_expanded_raw = load_jsonl(FILTERED_TRAIN_PATH)

print("Filtering out samples where the true answer was lost during retrieval...")
train_filtered_expanded = []
for row in train_filtered_expanded_raw:
    if row["answer"] in row["options"]:
        train_filtered_expanded.append(row)
print(f"Kept {len(train_filtered_expanded)} valid samples out of {len(train_filtered_expanded_raw)}.")

In [ ]:
print("Creating structural configuration...")
train_filtered_structural = make_structural(train_filtered_expanded)

print("Computing stratified domains...")
domains = [get_domain(row) for row in train_filtered_expanded]

print("Performing 85/15 train-validation split...")
train_idx, val_idx = train_test_split(
    range(len(train_filtered_expanded)), 
    test_size=0.15, 
    stratify=domains,
    random_state=42
)

# Split Full Info
train_full = [train_filtered_expanded[i] for i in train_idx]
val_full = [train_filtered_expanded[i] for i in val_idx]

# Split Structural
train_struct = [train_filtered_structural[i] for i in train_idx]
val_struct = [train_filtered_structural[i] for i in val_idx]

print(f"Saving splits (Train: {len(train_full)}, Val: {len(val_full)})...")
save_jsonl(train_full, f"{DATA_DIR}/train_filtered_top{TOP_K}_{safe_model_id}_fullInfo.jsonl")
save_jsonl(val_full, f"{DATA_DIR}/val_filtered_top{TOP_K}_{safe_model_id}_fullInfo.jsonl")
save_jsonl(train_struct, f"{DATA_DIR}/train_filtered_top{TOP_K}_{safe_model_id}_structOnly.jsonl")
save_jsonl(val_struct, f"{DATA_DIR}/val_filtered_top{TOP_K}_{safe_model_id}_structOnly.jsonl")

print("\u2705 Successfully generated train and validation files for both structOnly and fullInfo configurations.")